# Week 4: Chunking Experiments

**Target Limitations:**
- LIM-001: Text order mismatch (79.4% multi-column)
- LIM-005: No category filtering
- LIM-006: Common term contamination
- LIM-008: Retrieval gap for specific facts

**Strategies:**
- A (baseline): Recursive 1000/200
- B1 (small): Recursive 500/100
- B2 (large): Recursive 1500/300
- C3 (layout-aware): unstructured/docling/marker — 2hr time box
- C2 (custom sep): fallback if C3 fails

## 1. Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import time

load_dotenv()

PDF_DIR = Path("../data/raw_pdfs")
CHROMA_DIR = Path("../data/chroma_db")

# Test queries from Week 3
TEST_QUERIES = [
    {"query": "정수기 필터 교체는 어떻게 하나요?", "expected_category": "waterpurifier"},
    {"query": "공기청정기 필터 청소 방법 알려주세요", "expected_category": "airpurifier"},
    {"query": "청소기 배터리 충전 시간은 얼마나 되나요?", "expected_category": "unknown"},  # vacuum typo
    {"query": "Wi-Fi 연결이 안될 때 어떻게 해야 하나요?", "expected_category": None},  # cross-category
]

print(f"PDF directory: {PDF_DIR}")
print(f"PDFs: {list(PDF_DIR.glob('*.pdf'))}")

PDF directory: ../data/raw_pdfs
PDFs: [PosixPath('../data/raw_pdfs/waterpurifier_complex.pdf'), PosixPath('../data/raw_pdfs/airpurifier_simple.pdf'), PosixPath('../data/raw_pdfs/airpurifier_complex_MFL69726859_00_190321_00.pdf'), PosixPath('../data/raw_pdfs/vaccumcleaner_complex.pdf'), PosixPath('../data/raw_pdfs/waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf'), PosixPath('../data/raw_pdfs/vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf')]


## 2. Chunking Strategies

In [2]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def parse_filename(filename: str) -> dict:
    """Parse metadata from filename."""
    name = Path(filename).stem.lower()
    
    categories = ["waterpurifier", "airpurifier", "vacuumcleaner", "vaccumcleaner"]
    category = "unknown"
    for cat in categories:
        if cat in name:
            category = cat
            break
    
    complexity = "complex" if "complex" in name else "simple"
    
    return {"category": category, "complexity": complexity}

def chunk_with_strategy(pdf_path: Path, chunk_size: int, overlap: int) -> list[dict]:
    """Chunk a PDF with given parameters."""
    loader = PDFPlumberLoader(str(pdf_path))
    pages = loader.load()
    
    meta = parse_filename(pdf_path.name)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", "。", ".", " ", ""]
    )
    
    all_chunks = []
    for page in pages:
        page_num = page.metadata.get("page", 0) + 1
        chunks = splitter.split_text(page.page_content)
        
        for idx, chunk_text in enumerate(chunks):
            chunk_id = f"{meta['category']}_{meta['complexity']}_p{page_num:03d}_c{idx:03d}"
            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": pdf_path.name,
                    "category": meta["category"],
                    "complexity": meta["complexity"],
                    "page": page_num,
                    "chunk_id": chunk_id,
                    "chunk_index": idx,
                    "char_count": len(chunk_text),
                }
            })
    
    return all_chunks

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Strategy A — Baseline (1000/200)

In [ ]:
# Strategy A: Rebuild baseline with corrected category parsing
# (Week 3 had a bug: 'vaccumcleaner' typo not matched)

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import chromadb

# Delete old collection if exists
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection("lg_manuals")
    print("Deleted old lg_manuals collection")
except:
    print("No existing lg_manuals collection to delete")

# Rebuild with corrected parsing (uses parse_filename from cell-4 which has vaccumcleaner)
CHUNK_SIZE_A = 1000
CHUNK_OVERLAP_A = 200

all_chunks_A = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_strategy(pdf_path, CHUNK_SIZE_A, CHUNK_OVERLAP_A)
    all_chunks_A.extend(chunks)
    cat = chunks[0]["metadata"]["category"] if chunks else "?"
    print(f"  {pdf_path.name}: {len(chunks)} chunks (category: {cat})")

print(f"\nStrategy A total: {len(all_chunks_A)} chunks")

# Create embeddings and vectorstore
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
docs_A = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_A]

vectorstore_A = Chroma.from_documents(
    documents=docs_A,
    embedding=embeddings,
    collection_name="lg_manuals",
    persist_directory=str(CHROMA_DIR),
)

print(f"\nRebuilt baseline: {vectorstore_A._collection.count()} documents")

# Verify categories
results = vectorstore_A._collection.get(include=["metadatas"])
categories = {}
for meta in results["metadatas"]:
    cat = meta.get("category", "?")
    categories[cat] = categories.get(cat, 0) + 1
print(f"Categories: {categories}")

In [4]:
# Test retrieval with Strategy A
retriever_A = vectorstore_A.as_retriever(search_kwargs={"k": 5})

def evaluate_retrieval(retriever, queries):
    """Evaluate retrieval accuracy."""
    results = []
    for q in queries:
        docs = retriever.invoke(q["query"])
        correct = 0
        for doc in docs:
            cat = doc.metadata.get("category", "unknown")
            if q["expected_category"] is None:  # cross-category OK
                correct += 1
            elif cat == q["expected_category"]:
                correct += 1
        results.append({
            "query": q["query"][:30],
            "correct": correct,
            "total": len(docs),
            "accuracy": correct / len(docs) if docs else 0
        })
    return results

results_A = evaluate_retrieval(retriever_A, TEST_QUERIES)
print("Strategy A (baseline) Retrieval:")
for r in results_A:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Strategy A (baseline) Retrieval:
  정수기 필터 교체는 어떻게 하나요?... 3/5 (60%)
  공기청정기 필터 청소 방법 알려주세요... 4/5 (80%)
  청소기 배터리 충전 시간은 얼마나 되나요?... 3/5 (60%)
  Wi-Fi 연결이 안될 때 어떻게 해야 하나요?... 5/5 (100%)


## 4. Strategy B1 — Small Chunks (500/100)

In [5]:
# Strategy B1: Smaller chunks
CHUNK_SIZE_B1 = 500
CHUNK_OVERLAP_B1 = 100

all_chunks_B1 = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_strategy(pdf_path, CHUNK_SIZE_B1, CHUNK_OVERLAP_B1)
    all_chunks_B1.extend(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nStrategy B1 total: {len(all_chunks_B1)} chunks")
avg_size = sum(c['metadata']['char_count'] for c in all_chunks_B1) / len(all_chunks_B1)
print(f"Avg chunk size: {avg_size:.0f} chars")

  airpurifier_complex_MFL69726859_00_190321_00.pdf: 103 chunks
  airpurifier_simple.pdf: 96 chunks
  vaccumcleaner_complex.pdf: 120 chunks
  vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf: 44 chunks
  waterpurifier_complex.pdf: 98 chunks
  waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf: 78 chunks

Strategy B1 total: 539 chunks
Avg chunk size: 383 chars


In [6]:
# Create temp vector store for B1
from langchain_core.documents import Document

docs_B1 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_B1]

vectorstore_B1 = Chroma.from_documents(
    documents=docs_B1,
    embedding=embeddings,
    collection_name="lg_manuals_B1",
)

retriever_B1 = vectorstore_B1.as_retriever(search_kwargs={"k": 5})
results_B1 = evaluate_retrieval(retriever_B1, TEST_QUERIES)

print("Strategy B1 (500/100) Retrieval:")
for r in results_B1:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Strategy B1 (500/100) Retrieval:
  정수기 필터 교체는 어떻게 하나요?... 2/5 (40%)
  공기청정기 필터 청소 방법 알려주세요... 5/5 (100%)
  청소기 배터리 충전 시간은 얼마나 되나요?... 0/5 (0%)
  Wi-Fi 연결이 안될 때 어떻게 해야 하나요?... 5/5 (100%)


## 5. Strategy B2 — Large Chunks (1500/300)

In [18]:
# Strategy B2: Larger chunks
CHUNK_SIZE_B2 = 1500
CHUNK_OVERLAP_B2 = 300

all_chunks_B2 = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_strategy(pdf_path, CHUNK_SIZE_B2, CHUNK_OVERLAP_B2)
    all_chunks_B2.extend(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nStrategy B2 total: {len(all_chunks_B2)} chunks")
avg_size = sum(c['metadata']['char_count'] for c in all_chunks_B2) / len(all_chunks_B2)
print(f"Avg chunk size: {avg_size:.0f} chars")

  airpurifier_complex_MFL69726859_00_190321_00.pdf: 61 chunks
  airpurifier_simple.pdf: 53 chunks
  vaccumcleaner_complex.pdf: 54 chunks
  vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf: 24 chunks
  waterpurifier_complex.pdf: 43 chunks
  waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf: 34 chunks

Strategy B2 total: 269 chunks
Avg chunk size: 701 chars


In [19]:
# Create temp vector store for B2
docs_B2 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_B2]

vectorstore_B2 = Chroma.from_documents(
    documents=docs_B2,
    embedding=embeddings,
    collection_name="lg_manuals_B2",
)

retriever_B2 = vectorstore_B2.as_retriever(search_kwargs={"k": 5})
results_B2 = evaluate_retrieval(retriever_B2, TEST_QUERIES)

print("Strategy B2 (1500/300) Retrieval:")
for r in results_B2:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Strategy B2 (1500/300) Retrieval:
  정수기 필터 교체는 어떻게 하나요?... 3/5 (60%)
  공기청정기 필터 청소 방법 알려주세요... 4/5 (80%)
  청소기 배터리 충전 시간은 얼마나 되나요?... 0/5 (0%)
  Wi-Fi 연결이 안될 때 어떻게 해야 하나요?... 5/5 (100%)


## 6. Strategy C3 — Layout-Aware Parsing

**Time box: 2 hours**

**Evaluation order:**
1. `unstructured` — partition_pdf with strategy="hi_res"
2. `docling` — IBM's document understanding
3. `marker-pdf` — multi-column aware, outputs markdown

**Goal:** Address LIM-001 (79.4% multi-column pages)

**Decision criteria:**
- Text order preserved in multi-column pages
- Processing time reasonable
- Korean text quality maintained

### C3 Option 1: unstructured

NOT WORKING

In [14]:
# Try unstructured first
try:
    from unstructured.partition.pdf import partition_pdf
    UNSTRUCTURED_AVAILABLE = True
    print("unstructured available")
except ImportError:
    UNSTRUCTURED_AVAILABLE = False
    print("unstructured not available — install with: uv add unstructured[pdf]")

unstructured available


In [19]:
# Test unstructured on sample PDF (if available)
if UNSTRUCTURED_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    print(f"Testing unstructured on: {sample_pdf.name}")
    
    start_time = time.time()
    elements = partition_pdf(
        filename=str(sample_pdf),
        languages=["ko"],
        strategy="fast",  # Start with fast, upgrade to hi_res if needed
    )
    elapsed = time.time() - start_time
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Elements: {len(elements)}")
    print(f"Element types: {set(type(e).__name__ for e in elements)}")
    
    # Show sample elements
    for e in elements[:5]:
        print(f"  [{type(e).__name__}] {str(e)[:100]}...")
else:
    print("Skipping unstructured test")

Testing unstructured on: waterpurifier_complex.pdf
Time: 0.5s
Elements: 0
Element types: set()


In [18]:
# Check multi-column handling with hi_res strategy
if UNSTRUCTURED_AVAILABLE:
    # Test specific page known to have multi-column (page 11 from Week 3)
    print("Testing hi_res strategy for multi-column...")
    print("(This may take longer)")
    
    start_time = time.time()
    elements_hires = partition_pdf(
        filename=str(sample_pdf),
        languages=["ko"],
        strategy="hi_res",
        infer_table_structure=True,
    )
    elapsed = time.time() - start_time
    
    print(f"Time (hi_res): {elapsed:.1f}s")
    print(f"Elements: {len(elements_hires)}")
else:
    print("Skipping hi_res test")

Testing hi_res strategy for multi-column...
(This may take longer)


TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

### C3 Option 2: docling

NOT WORKING

IBM's document understanding. Install: `uv add docling`

In [12]:
# Try docling
try:
    from docling.document_converter import DocumentConverter
    DOCLING_AVAILABLE = True
    print("docling available")
except ImportError:
    DOCLING_AVAILABLE = False
    print("docling not available — install with: uv add docling")

docling available


In [13]:
# Test docling if available
if DOCLING_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    print(f"Testing docling on: {sample_pdf.name}")
    
    start_time = time.time()
    converter = DocumentConverter()
    result = converter.convert(str(sample_pdf))
    elapsed = time.time() - start_time
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Document: {result.document}")
else:
    print("Skipping docling test")

Testing docling on: waterpurifier_complex.pdf


[INFO] 2026-05-18 07:49:48,239 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-18 07:49:48,245 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-18 07:49:49,347 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-05-18 07:49:49,738 [RapidOCR] download_file.py:95: Successfully saved to: /Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-18 07:49:49,742 [RapidOCR] main.py:57: Using /Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-18 07:49:49,832 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-18 07:49:49,833 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolv

Time: 78.3s
Document: schema_name='DoclingDocument' version='1.10.0' name='waterpurifier_complex' origin=DocumentOrigin(mimetype='application/pdf', binary_hash=15403382494471836180, filename='waterpurifier_complex.pdf', uri=None) furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) body=GroupItem(self_ref='#/body', parent=None, children=[RefItem(cref='#/pictures/0'), RefItem(cref='#/texts/0'), RefItem(cref='#/texts/1'), RefItem(cref='#/texts/2'), RefItem(cref='#/texts/3'), RefItem(cref='#/texts/4'), RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/texts/9'), RefItem(cref='#/pictures/1'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/groups/0'), RefItem(cref='#/pictures/2'), RefItem(cref='#/pictures/3'), RefItem(cref='#/pictures/4'), RefItem(cref=

### C3 Option 3: pymupdf4llm

PyMuPDF-based markdown converter with layout detection. Handles multi-column layouts.

Install: `uv add pymupdf4llm`

In [8]:
# Try pymupdf4llm
try:
    import pymupdf4llm
    PYMUPDF4LLM_AVAILABLE = True
    print("pymupdf4llm available")
except ImportError:
    PYMUPDF4LLM_AVAILABLE = False
    print("pymupdf4llm not available — install with: uv add pymupdf4llm")

pymupdf4llm available


In [9]:
# Test pymupdf4llm if available
if PYMUPDF4LLM_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    print(f"Testing pymupdf4llm on: {sample_pdf.name}")
    
    start_time = time.time()
    md_text = pymupdf4llm.to_markdown(str(sample_pdf))
    elapsed = time.time() - start_time
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Markdown output length: {len(md_text)} chars")
    print(f"\nSample output (first 1500 chars):")
    print("-" * 50)
    print(md_text[:1500])
else:
    print("Skipping pymupdf4llm test")

Testing pymupdf4llm on: waterpurifier_complex.pdf
Time: 4.5s
Markdown output length: 42306 chars

Sample output (first 1500 chars):
--------------------------------------------------
**==> picture [97 x 44] intentionally omitted <==**

## 제품 사용설명서 데스크 정수기 

제품을 안전하고 편리하게 사용하기 위해 반드시 제품을 사용하기 전에 사용설명서를 읽어주세요. 제품 보증서도 함께 들어있으니 잘 보관하세요. 

본 제품은 가정에서만 사용하는 실내용 기기입니다. 상업용, 실험용, 산업용으로 사용하지 마세요. 

## 권장 안전 사용 기간 : 7년 

권장 안전 사용 기간을 초과해서 사용하면 사용 환경의 변화나 제품의 노후로 인해 안전사고가 발생할 수 있습니다. 권장 안전 사용 기간 내에 안전 점검을 받으세요. (안전 점검은 유료 서비스입니다.) 

## 전문 기술이 필요한 제품을 설치할 때는 LG전자 서비스 센터를 이용하세요. 

제품을 잘못 설치해서 생긴 문제는 설치한 사람의 책임이며, 이런 경우 제품 보증 기간 안에도 무상 서비스를 받을 수 없습니다. 

모델명 : WD523A** / WD524A** / WD520A** / WD521A** / WD321A** / WD323A** / WD507A** / WD508A** 

**==> picture [218 x 56] intentionally omitted <==**

MFL71928101 Rev.15_030626 

## 안전을 위해 주의하기 

- 3 제품을 사용하기 전에 읽어주세요. 3 경고 7 주의 

- 38 폐가전제품 처리 절차 38 오픈소스 정보 38 제품 규격 39 생활 속 전기안전 캠페인 

## LG ThinQ 사용하기 

- 9 LG ThinQ 와 LG 가전 연결하기 

- 9 LG ThinQ 앱으로 정수기

In [10]:
# Compare multi-column handling: pymupdf4llm vs baseline (PDFPlumber)
# Pick a known multi-column page to compare text order

if PYMUPDF4LLM_AVAILABLE:
    sample_pdf = list(PDF_DIR.glob("waterpurifier_complex.pdf"))[0]
    
    # Get page 11 (known multi-column from Week 3 analysis)
    test_page = 10  # 0-indexed
    
    # Baseline: PDFPlumber
    loader = PDFPlumberLoader(str(sample_pdf))
    pages_baseline = loader.load()
    baseline_text = pages_baseline[test_page].page_content if len(pages_baseline) > test_page else "N/A"
    
    # pymupdf4llm: single page
    md_pages = pymupdf4llm.to_markdown(str(sample_pdf), pages=[test_page])
    
    print(f"=== Page {test_page + 1} Comparison ===\n")
    print("--- BASELINE (PDFPlumber) ---")
    print(baseline_text[:800])
    print("\n" + "=" * 50 + "\n")
    print("--- PYMUPDF4LLM ---")
    print(md_pages[:800])

=== Page 11 Comparison ===

--- BASELINE (PDFPlumber) ---
LG ThinQ 사용하기 11
2
제품의 3 m 거리 이내에서 ‘Hi LG’(하이 엘지)를
기능 명령어 (예시)
말하세요.
오늘/일주일간/이번 달의 물 사용량을
• 작동음이 울리고 음성 명령을 들을 준비 상태가 되면 확인할 수 있습니다.
물 사용량
제어창의 x 아이콘이 깜빡입니다. 조회 • 오늘 물 사용량 알려줘
• 이번 주 냉수 사용량 알려줘
• 음성 인식 기능 뭐 있어?
안내 • 음성 인식 어떻게 사용해?
• LG전자 서비스 센터 번호 알려줘
3
명령어를 말해보세요. • 날씨 알려줘
기타 • 지금 몇시야?
• 사용자가 요청한 명령을 수행하고, 결과에 대한 안내
음성을 제공합니다. • 너 이름이 뭐야?
알아두기
• 어린이의 음성의 경우 음성 인식이 작동하지 않을 수
있습니다.
• 사용자의 연령, 목소리 크기, 억양, 주변 환경소음, 사용
알아두기 환경에 따라 음성 인식의 차이가 발생할 수 있습니다.
• 준비 상태에서 일정 시간 (6초) 동안 명령어를 말하지 • 사용자와 제품 사이에 장애물이 있거나 제품이 개방되지
않으면 준비 상태가 해제됩니다. 해제된 경우에는 ‘Hi 않은 환경에 설치되어 있으면 음성 인식의 차이가
LG’ (하이 엘지)를 다시 말하세요. 발생할 수 있습니다.
• 명령어를 인식하지 못한 경우, ‘Hi LG’(하이 엘지)를 • 음성 인식률을 높이기 위해서는 제품으로부터 3 m
제외하고 명령어만 다시 말하세요. 이내의 조용한 환경에서 사용하세요.
• 네트워크 연결 상태에 따라 명령을 수행하는 시간이 • 온수 출수 등 일부 기능은 음성 인식으로 사용할 수
다소 걸릴 수 있습니다. 없습니다.
• 사용


--- PYMUPDF4LLM ---
LG ThinQ 사용하기 **11** 

- **2** 제품의 3 m 거리 이내에서 ‘Hi LG’(하이 엘지)를 말하세요. 

   - 작동음이 울리고 음성 명령을 들을 준비 상태가 되면 제어창의 x 아이콘이 깜빡

In [ ]:
# Strategy C3: Full chunking with pymupdf4llm
import re

def clean_markdown(text: str) -> str:
    """Remove markdown artifacts for cleaner embeddings."""
    # Remove image placeholders
    text = re.sub(r'\*\*==> picture.*?<==\*\*', '', text)
    # Remove bold/italic markers
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    # Simplify table separators
    text = re.sub(r'\|[-:]+\|', '', text)
    # Remove <br> tags
    text = text.replace('<br>', ' ')
    # Clean up multiple newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def chunk_with_pymupdf4llm(pdf_path: Path, chunk_size: int = 1000, overlap: int = 200) -> list[dict]:
    """Chunk PDF using pymupdf4llm markdown output (cleaned)."""
    md_text = pymupdf4llm.to_markdown(str(pdf_path))
    
    # Clean markdown artifacts
    cleaned_text = clean_markdown(md_text)
    
    meta = parse_filename(pdf_path.name)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    chunks = splitter.split_text(cleaned_text)
    
    all_chunks = []
    for idx, chunk_text in enumerate(chunks):
        chunk_id = f"{meta['category']}_{meta['complexity']}_c{idx:03d}"
        all_chunks.append({
            "text": chunk_text,
            "metadata": {
                "source": pdf_path.name,
                "category": meta["category"],
                "complexity": meta["complexity"],
                "chunk_id": chunk_id,
                "chunk_index": idx,
                "char_count": len(chunk_text),
            }
        })
    
    return all_chunks

if PYMUPDF4LLM_AVAILABLE:
    all_chunks_C3 = []
    for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
        print(f"Processing {pdf_path.name}...", end=" ")
        start = time.time()
        chunks = chunk_with_pymupdf4llm(pdf_path)
        all_chunks_C3.extend(chunks)
        print(f"{len(chunks)} chunks ({time.time() - start:.1f}s)")
    
    print(f"\nStrategy C3 total: {len(all_chunks_C3)} chunks")
    avg_size = sum(c['metadata']['char_count'] for c in all_chunks_C3) / len(all_chunks_C3)
    print(f"Avg chunk size: {avg_size:.0f} chars")
    
    # Show sample cleaned chunk
    print(f"\nSample cleaned chunk (first 500 chars):")
    print("-" * 50)
    print(all_chunks_C3[5]["text"][:500])
else:
    all_chunks_C3 = []
    print("Skipping C3 — pymupdf4llm not available")

In [ ]:
# Create vector store for C3 and evaluate
from langchain_core.documents import Document  # ensure import

if PYMUPDF4LLM_AVAILABLE and all_chunks_C3:
    docs_C3 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_C3]
    
    vectorstore_C3 = Chroma.from_documents(
        documents=docs_C3,
        embedding=embeddings,
        collection_name="lg_manuals_C3",
    )
    
    retriever_C3 = vectorstore_C3.as_retriever(search_kwargs={"k": 5})
    results_C3 = evaluate_retrieval(retriever_C3, TEST_QUERIES)
    
    print("Strategy C3 (pymupdf4llm cleaned) Retrieval:")
    for r in results_C3:
        print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")
else:
    results_C3 = None
    print("Skipping C3 evaluation")

## 7. Strategy C2 — Custom Separators (Fallback)

If C3 (layout-aware) fails or takes too long, use custom separators:
- Split by section headers
- Split by step numbers
- Preserve table boundaries

In [13]:
# Custom separators based on Korean manual patterns
KOREAN_SEPARATORS = [
    "\n\n\n",           # Triple newline (section break)
    "\n\n",             # Double newline (paragraph break)
    r"\n\d+\s",         # Step numbers ("1 ", "2 ")
    "\n•\s",            # Bullet points
    "\n- ",             # Dash lists
    "\n",               # Single newline
    "。",               # Korean period
    ".",                # Period
    " ",                # Space
    "",                 # Character
]

def chunk_with_custom_sep(pdf_path: Path, chunk_size: int = 1000, overlap: int = 200) -> list[dict]:
    """Chunk with Korean-specific separators."""
    loader = PDFPlumberLoader(str(pdf_path))
    pages = loader.load()
    
    meta = parse_filename(pdf_path.name)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=KOREAN_SEPARATORS,
    )
    
    all_chunks = []
    for page in pages:
        page_num = page.metadata.get("page", 0) + 1
        chunks = splitter.split_text(page.page_content)
        
        for idx, chunk_text in enumerate(chunks):
            chunk_id = f"{meta['category']}_{meta['complexity']}_p{page_num:03d}_c{idx:03d}"
            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": pdf_path.name,
                    "category": meta["category"],
                    "complexity": meta["complexity"],
                    "page": page_num,
                    "chunk_id": chunk_id,
                    "chunk_index": idx,
                    "char_count": len(chunk_text),
                }
            })
    
    return all_chunks

In [14]:
# Test C2 on all PDFs
all_chunks_C2 = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = chunk_with_custom_sep(pdf_path)
    all_chunks_C2.extend(chunks)
    print(f"  {pdf_path.name}: {len(chunks)} chunks")

print(f"\nStrategy C2 total: {len(all_chunks_C2)} chunks")
avg_size = sum(c['metadata']['char_count'] for c in all_chunks_C2) / len(all_chunks_C2)
print(f"Avg chunk size: {avg_size:.0f} chars")

  airpurifier_complex_MFL69726859_00_190321_00.pdf: 67 chunks
  airpurifier_simple.pdf: 58 chunks
  vaccumcleaner_complex.pdf: 74 chunks
  vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf: 26 chunks
  waterpurifier_complex.pdf: 57 chunks
  waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf: 45 chunks

Strategy C2 total: 327 chunks
Avg chunk size: 591 chars


In [15]:
# Create vector store for C2
docs_C2 = [Document(page_content=c["text"], metadata=c["metadata"]) for c in all_chunks_C2]

vectorstore_C2 = Chroma.from_documents(
    documents=docs_C2,
    embedding=embeddings,
    collection_name="lg_manuals_C2",
)

retriever_C2 = vectorstore_C2.as_retriever(search_kwargs={"k": 5})
results_C2 = evaluate_retrieval(retriever_C2, TEST_QUERIES)

print("Strategy C2 (custom sep) Retrieval:")
for r in results_C2:
    print(f"  {r['query']}... {r['correct']}/{r['total']} ({r['accuracy']:.0%})")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Strategy C2 (custom sep) Retrieval:
  정수기 필터 교체는 어떻게 하나요?... 3/5 (60%)
  공기청정기 필터 청소 방법 알려주세요... 4/5 (80%)
  청소기 배터리 충전 시간은 얼마나 되나요?... 0/5 (0%)
  Wi-Fi 연결이 안될 때 어떻게 해야 하나요?... 5/5 (100%)


## 8. Comparison Summary

In [20]:
# Build comparison table
def avg_accuracy(results):
    return sum(r['accuracy'] for r in results) / len(results)

comparison = [
    {"strategy": "A (baseline 1000/200)", "chunks": vectorstore_A._collection.count(), "avg_accuracy": avg_accuracy(results_A)},
    {"strategy": "B1 (small 500/100)", "chunks": len(all_chunks_B1), "avg_accuracy": avg_accuracy(results_B1)},
    {"strategy": "B2 (large 1500/300)", "chunks": len(all_chunks_B2), "avg_accuracy": avg_accuracy(results_B2)},
    {"strategy": "C2 (custom sep)", "chunks": len(all_chunks_C2), "avg_accuracy": avg_accuracy(results_C2)},
]

# Add C3 if available
if results_C3:
    comparison.append({"strategy": "C3 (pymupdf4llm)", "chunks": len(all_chunks_C3), "avg_accuracy": avg_accuracy(results_C3)})

print(f"{'Strategy':<25} {'Chunks':<10} {'Avg Accuracy':<15}")
print("-" * 50)
for c in comparison:
    print(f"{c['strategy']:<25} {c['chunks']:<10} {c['avg_accuracy']:.0%}")

Strategy                  Chunks     Avg Accuracy   
--------------------------------------------------
A (baseline 1000/200)     320        75%
B1 (small 500/100)        539        60%
B2 (large 1500/300)       269        60%
C2 (custom sep)           327        60%
C3 (pymupdf4llm)          339        65%


## 9. Qualitative Analysis: LIM-005/008 Queries

In [21]:
# Compare retrieval for LIM-005 query across strategies
query_005 = "정수기 필터 교체는 어떻게 하나요?"

print(f"Query: {query_005}")
print("=" * 70)

retrievers = [("A", retriever_A), ("B1", retriever_B1), ("B2", retriever_B2), ("C2", retriever_C2)]
if results_C3:
    retrievers.append(("C3", retriever_C3))

for name, retriever in retrievers:
    docs = retriever.invoke(query_005)
    categories = [d.metadata.get('category', '?') for d in docs]
    correct = sum(1 for c in categories if c == 'waterpurifier')
    print(f"\n[{name}] {correct}/5 correct")
    print(f"    Categories: {categories}")

Query: 정수기 필터 교체는 어떻게 하나요?

[A] 3/5 correct
    Categories: ['waterpurifier', 'airpurifier', 'waterpurifier', 'airpurifier', 'waterpurifier']

[B1] 2/5 correct
    Categories: ['waterpurifier', 'airpurifier', 'airpurifier', 'waterpurifier', 'airpurifier']

[B2] 3/5 correct
    Categories: ['waterpurifier', 'airpurifier', 'waterpurifier', 'airpurifier', 'waterpurifier']

[C2] 3/5 correct
    Categories: ['waterpurifier', 'airpurifier', 'waterpurifier', 'airpurifier', 'waterpurifier']

[C3] 4/5 correct
    Categories: ['waterpurifier', 'waterpurifier', 'airpurifier', 'waterpurifier', 'waterpurifier']


In [22]:
# Compare retrieval for LIM-008 query across strategies
query_008 = "청소기 배터리 충전 시간은 얼마나 되나요?"

print(f"Query: {query_008}")
print("=" * 70)

retrievers = [("A", retriever_A), ("B1", retriever_B1), ("B2", retriever_B2), ("C2", retriever_C2)]
if results_C3:
    retrievers.append(("C3", retriever_C3))

for name, retriever in retrievers:
    docs = retriever.invoke(query_008)
    
    print(f"\n[{name}]")
    for i, doc in enumerate(docs):
        has_charging = '충전' in doc.page_content
        has_time = '시간' in doc.page_content or '분' in doc.page_content
        cat = doc.metadata.get('category', '?')
        print(f"  [{i+1}] {cat} | 충전: {'✓' if has_charging else '✗'} | 시간/분: {'✓' if has_time else '✗'}")

Query: 청소기 배터리 충전 시간은 얼마나 되나요?

[A]
  [1] unknown | 충전: ✓ | 시간/분: ✓
  [2] unknown | 충전: ✓ | 시간/분: ✓
  [3] unknown | 충전: ✓ | 시간/분: ✓
  [4] airpurifier | 충전: ✗ | 시간/분: ✓
  [5] airpurifier | 충전: ✗ | 시간/분: ✓

[B1]
  [1] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [2] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [3] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [4] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [5] vaccumcleaner | 충전: ✗ | 시간/분: ✓

[B2]
  [1] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [2] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [3] airpurifier | 충전: ✗ | 시간/분: ✓
  [4] airpurifier | 충전: ✗ | 시간/분: ✓
  [5] waterpurifier | 충전: ✗ | 시간/분: ✓

[C2]
  [1] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [2] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [3] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [4] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [5] vaccumcleaner | 충전: ✗ | 시간/분: ✓

[C3]
  [1] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [2] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [3] vaccumcleaner | 충전: ✓ | 시간/분: ✗
  [4] vaccumcleaner | 충전: ✓ | 시간/분: ✓
  [5] airpurifier | 충전: ✗ | 시간/분: ✓


## 10. Findings

### C3 (Layout-aware) Status
| Library | Status | Notes |
|---------|--------|-------|
| unstructured (fast) | ❌ Failed | 0 elements extracted from Korean PDF |
| unstructured (hi_res) | ❌ Skipped | Requires tesseract/poppler, hung on execution |
| docling | ⚠️ Partial | UTF-8 errors on pages 36-39, 78s processing |
| marker-pdf | ❌ N/A | Dependency collision, couldn't install |
| **pymupdf4llm** | ✅ Used | Layout-aware markdown + cleaning |

### Baseline Bug (LIM-003 related)
- Week 3 regex: `r'^(waterpurifier|airpurifier|vacuumcleaner)'` (single 'c')
- Actual filenames: `vaccumcleaner_*.pdf` (double 'c' typo)
- Result: 95 vacuum cleaner chunks categorized as "unknown"
- **Baseline A's 75% accuracy is artificial** — comparison should use B1/B2/C2/C3 only

### Comparison Results (excluding broken baseline)
| Strategy | Chunks | Avg Accuracy | Best For |
|----------|--------|--------------|----------|
| B1 (500/100) | 539 | 60% | LIM-008 (specific facts) |
| B2 (1500/300) | 269 | 60% | — |
| C2 (custom sep) | 327 | 60% | LIM-008 (specific facts) |
| **C3 (pymupdf4llm)** | 339 | **65%** | **LIM-005 (category precision)** |

### LIM-005: 정수기 필터 교체 (Category Precision)
- **C3 wins: 4/5** vs 3/5 for others
- Layout-aware parsing preserves document structure → better category separation
- Markdown cleaning (removing `**bold**`, image placeholders) was critical

### LIM-008: 청소기 배터리 충전 시간 (Specific Fact Retrieval)
- **B1 and C2 win: 5/5** vacuum category
- Smaller chunks (B1) or custom separators (C2) help isolate specific facts
- C3: 4/5 — competitive but not best

### Key Insight
**No single chunking strategy wins all tasks:**
- Layout-aware (C3) → category-specific queries
- Small chunks (B1) / custom separators (C2) → specific fact retrieval
- **Implication for Week 5:** Metadata filtering or hybrid approach may combine strengths

### Recommendations
1. Fix filename typo (`vaccumcleaner` → `vacuumcleaner`) or update regex permanently
2. Consider **hybrid approach**: pymupdf4llm for parsing + custom separators for chunking
3. Week 5: Add metadata filtering to address LIM-005/006 at retrieval time